Voting Classifier

In [13]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# === 1. Wczytanie danych ===
df = pd.read_csv("nba_dataset_2010_2023.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2023
test_mask = df["season"] == 2023

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna (czy dostanie nagrodę)
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Przygotowanie danych dla Stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}

X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1]
y_train_stage2_mapped = y_train_stage2.map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. VotingClassifier: XGB + GBC + LogReg (ze skalowaniem)
model_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

model_gbc = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)

voting_model = VotingClassifier(
    estimators=[
        ("xgb", model_xgb),
        ("gbc", model_gbc),
        ("logreg", model_logreg)
    ],
    voting="soft"
)

# === 5. Trening i predykcja stage2
voting_model.fit(X_train_stage2, y_train_stage2_mapped)
probas_stage2 = voting_model.predict_proba(X_test_stage2)

# === 6. Tworzenie DataFrame i generowanie piątek
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 7. Zapis do JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ VotingClassifier z poprawnym skalowaniem LogReg działa!")


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:40:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:40:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ VotingClassifier z poprawnym skalowaniem LogReg działa!


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


StackingClassifier

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# === 1. Wczytanie danych ===
df = pd.read_csv("nba_dataset_2010_2024.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2024
test_mask = df["season"] == 2024

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: binary classification (nagroda?)
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Przygotowanie danych dla Stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. StackingClassifier z LogisticRegression + StandardScaler
estimators = [
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    )),
    ("gbc", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
]

final_estimator = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. Trening + predykcja
stacking_model.fit(X_train_stage2, y_train_stage2)
probas_stage2 = stacking_model.predict_proba(X_test_stage2)

# === 6. Zbudowanie piątek (TOP 15 wg max z klas 1–3)
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

# Oblicz maksymalne prawdopodobieństwo All-NBA
df_pred["all_nba_score"] = df_pred[[1, 2, 3]].max(axis=1)

# Wybierz 15 najlepszych i rozdziel na piątki
top15_allnba = df_pred.sort_values("all_nba_score", ascending=False).head(15)
results = {
    "first all-nba team": top15_allnba.iloc[:5]["Player"].tolist(),
    "second all-nba team": top15_allnba.iloc[5:10]["Player"].tolist(),
    "third all-nba team": top15_allnba.iloc[10:15]["Player"].tolist()
}

# Rookie piątki tak jak wcześniej (bez powtórek)
ordinal = ["first", "second"]
already_rookies = set()
for idx, class_id in enumerate([4, 5]):
    rookies = df_pred[(df_pred["is_rookie"] == 1) & (~df_pred["Player"].isin(already_rookies))]
    top5_rookies = rookies.sort_values(class_id, ascending=False).head(5)
    results[f"{ordinal[idx]} rookie all-nba team"] = top5_rookies["Player"].tolist()
    already_rookies.update(top5_rookies["Player"])



# === 7. Zapis do JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅")


<<<<<<< local <modified: >


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [10:29:18] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [10:29:20] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[10:29:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:29:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:29:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.



/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:22:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:22:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[13:22:24] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[13:22:24] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[13:22:24] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.



>>>>>>> remote <modified: >


✅


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [19]:
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2


In [20]:
# Zapis pełnych prawdopodobieństw dla graczy z stage2
df_pred_sorted = df_pred.sort_values(by=1, ascending=False)  # możesz też sortować po max(axis=1)
df_pred_sorted.to_csv("stage2_all_players_probs.csv", index=False)


In [9]:
len(X_test_stage2)

25

In [11]:
# === 📦 Stage 1: prawdopodobieństwa (wszyscy gracze sezonu 2023)
stage1_probas = model_bin.predict_proba(X_test)[:, 1]  # prawdopodobieństwo klasy 1 (nagroda)

df_stage1_probs = pd.DataFrame({
    "Player": players_test,
    "is_rookie": is_rookie,
    "prob_has_award": stage1_probas
}).sort_values("prob_has_award", ascending=False)
df_stage1_probs.to_csv("stage1_probs.csv", index=False)

# === 📦 Stage 2: prawdopodobieństwa klas 1–5 (tylko stage1_preds == 1)
df_stage2_probs = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_stage2_probs["Player"] = players_stage2
df_stage2_probs["is_rookie"] = is_rookie_stage2
df_stage2_probs["max_prob"] = df_stage2_probs[[1, 2, 3, 4, 5]].max(axis=1)
df_stage2_probs = df_stage2_probs.sort_values("max_prob", ascending=False)
df_stage2_probs.drop(columns=["max_prob"], inplace=True)
df_stage2_probs.to_csv("stage2_probs.csv", index=False)

RandomizedSearchCV StackingClassifier

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV

# === 1. Dane
df = pd.read_csv("nba_dataset_2010_2024.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)
train_mask = df["season"] < 2024
test_mask = df["season"] == 2024

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(objective="binary:logistic", use_label_encoder=False, eval_metric="logloss", random_state=42)
sample_weight = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight)
stage1_preds = model_bin.predict(X_test)

# === 3. Stage 2: dane dla klas 1–5
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

# === 4. Stack model
xgb = XGBClassifier(objective="multi:softprob", num_class=5, use_label_encoder=False, eval_metric="mlogloss", random_state=42)
gbc = GradientBoostingClassifier(random_state=42)
logreg_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", multi_class="multinomial", solver="lbfgs", random_state=42))

stack = StackingClassifier(
    estimators=[("xgb", xgb), ("gbc", gbc)],
    final_estimator=logreg_pipe,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. Parametry do strojenia
param_dist = {
  "xgb__n_estimators": [180, 200, 220],
  "xgb__max_depth": [2, 3, 4],
  "xgb__learning_rate": [0.08, 0.1, 0.12],
  "gbc__n_estimators": [180, 200, 220],
  "gbc__max_depth": [4, 5, 6],
  "gbc__learning_rate": [0.18, 0.2, 0.22],
  "final_estimator__logisticregression__C": [5, 10, 15]
}


# === 6. Randomized Search
search = RandomizedSearchCV(
    estimator=stack,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1,
    random_state=42
)
search.fit(X_train_stage2, y_train_stage2)
best_model = search.best_estimator_

# === 7. Predykcja stage2
probas_stage2 = best_model.predict_proba(X_test_stage2)

# === 8. Selekcja piątek (top15 → optymalne przypisanie)
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2
df_pred["all_nba_score"] = df_pred[[1, 2, 3]].sum(axis=1)

top15 = df_pred.sort_values("all_nba_score", ascending=False).head(15).copy()
ranks = {
    "first all-nba team": top15.sort_values(1, ascending=False)["Player"].tolist(),
    "second all-nba team": top15.sort_values(2, ascending=False)["Player"].tolist(),
    "third all-nba team": top15.sort_values(3, ascending=False)["Player"].tolist()
}
results = {team: [] for team in ranks}
used_players = set()

while any(len(results[team]) < 5 for team in results):
    for team in ["first all-nba team", "second all-nba team", "third all-nba team"]:
        for player in ranks[team]:
            if player not in used_players:
                results[team].append(player)
                used_players.add(player)
                break

# === 9. Rookie teams
ordinal = ["first", "second"]
already_rookies = set()
for idx, class_id in enumerate([4, 5]):
    rookies = df_pred[(df_pred["is_rookie"] == 1) & (~df_pred["Player"].isin(already_rookies))]
    top5 = rookies.sort_values(class_id, ascending=False).head(5)
    results[f"{ordinal[idx]} rookie all-nba team"] = top5["Player"].tolist()
    already_rookies.update(top5["Player"])

# === 10. Zapis JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ Gotowe: zapisano classification_result.json")


In [23]:
print(best_model)

StackingClassifier(cv=3,
                   estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                    

In [25]:
with open("best_params_stage2_stacking.json", "w") as f2:
    json.dump(search.best_params_, f2, indent=2)


In [34]:
search.best_params_

{'xgb__n_estimators': 180,
 'xgb__max_depth': 3,
 'xgb__learning_rate': 0.12,
 'gbc__n_estimators': 220,
 'gbc__max_depth': 5,
 'gbc__learning_rate': 0.2,
 'final_estimator__logisticregression__C': 15}

In [35]:
# Zapis pełnych prawdopodobieństw dla graczy z stage2
df_pred_sorted = df_pred.sort_values(by=1, ascending=False)  # możesz też sortować po max(axis=1)
df_pred_sorted.to_csv("stage2_all_players_probs.csv", index=False)

Stacking z najlepszymi parametrami

In [39]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

# === Bazowe modele z best_params
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

gbc = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.2,
    random_state=42
)

logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        C=10,
        random_state=42
    )
)

# === 1. Dane
df = pd.read_csv("nba_dataset_2010_2024.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)
train_mask = df["season"] < 2024
test_mask = df["season"] == 2024

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)
y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(objective="binary:logistic", use_label_encoder=False, eval_metric="logloss", random_state=42)
sample_weight = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight)
stage1_preds = model_bin.predict(X_test)

# === 3. Stage 2: dane dla klas 1–5
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

# === 4. Stack model


# === Finalny stacking model
stacking_best = StackingClassifier(
    estimators=[("xgb", xgb), ("gbc", gbc)],
    final_estimator=logreg_pipe,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)



stacking_best.fit(X_train_stage2, y_train_stage2)

# === 7. Predykcja stage2
probas_stage2 = stacking_best.predict_proba(X_test_stage2)

# === 8. Selekcja piątek (top15 → optymalne przypisanie)
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2
df_pred["all_nba_score"] = df_pred[[1, 2, 3]].sum(axis=1)

top15 = df_pred.sort_values("all_nba_score", ascending=False).head(15).copy()
ranks = {
    "first all-nba team": top15.sort_values(1, ascending=False)["Player"].tolist(),
    "second all-nba team": top15.sort_values(2, ascending=False)["Player"].tolist(),
    "third all-nba team": top15.sort_values(3, ascending=False)["Player"].tolist()
}
results = {team: [] for team in ranks}
used_players = set()

while any(len(results[team]) < 5 for team in results):
    for team in ["first all-nba team", "second all-nba team", "third all-nba team"]:
        for player in ranks[team]:
            if player not in used_players:
                results[team].append(player)
                used_players.add(player)
                break

# === 9. Rookie teams — iteracyjna selekcja jak dla All-NBA
rookies_df = df_pred[df_pred["is_rookie"] == 1].copy()

# Rankingi rookie wg P(4) i P(5)
rookie_ranks = {
    "first rookie all-nba team": rookies_df.sort_values(4, ascending=False)["Player"].tolist(),
    "second rookie all-nba team": rookies_df.sort_values(5, ascending=False)["Player"].tolist()
}

# Przydziel unikalnych graczy do rookie piątek
results.update({team: [] for team in rookie_ranks})
used_rookies = set()

while any(len(results[team]) < 5 for team in rookie_ranks):
    for team in ["first rookie all-nba team", "second rookie all-nba team"]:
        for player in rookie_ranks[team]:
            if player not in used_rookies:
                results[team].append(player)
                used_rookies.add(player)
                break

# === 10. Zapis JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ Gotowe: zapisano classification_result.json")


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:42:05] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:42:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[15:42:12] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[15:42:12] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[15:42:12] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.



✅ Gotowe: zapisano classification_result.json


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [40]:
# Zapis pełnych prawdopodobieństw dla graczy z stage2
df_pred_sorted = df_pred.sort_values(by=1, ascending=False)  # możesz też sortować po max(axis=1)
df_pred_sorted.to_csv("stage2_all_players_probs.csv", index=False)

In [2]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV

# === 1. Wczytanie danych ===
df = pd.read_csv("nba_dataset_2010_2023.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2023
test_mask = df["season"] == 2023

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Przygotowanie danych dla Stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}

X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. Składniki stacking
model_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

model_gbc = GradientBoostingClassifier(random_state=42)

logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)

stacking_model = StackingClassifier(
    estimators=[
        ("xgb", model_xgb),
        ("gbc", model_gbc)
    ],
    final_estimator=logreg_pipeline,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. Rozszerzona przestrzeń hiperparametrów
param_distributions = {
    "final_estimator__logisticregression__C": [0.001, 0.01, 0.1, 1, 5, 10],
    "gbc__n_estimators": [100, 200, 300, 500],
    "gbc__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gbc__max_depth": [3, 4, 5, 6],
    "gbc__subsample": [0.7, 0.8, 1.0],
    "xgb__n_estimators": [200, 300, 500],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 1, 5]
}

search = RandomizedSearchCV(
    estimator=stacking_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train_stage2, y_train_stage2)
best_model = search.best_estimator_

# === 6. Predykcja
probas_stage2 = best_model.predict_proba(X_test_stage2)

df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 7. Zapis wyniku
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# print("✅ Zapisano: stacking_deep_tuning_result.json")
# print("✅ Najlepsze parametry:")
# print(search.best_params_)


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [10:28:50] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[10:28:54] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:54] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:54] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[10:28:5

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# === 1. Wczytanie danych ===
df = pd.read_csv("nba_dataset_2010_2024.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)

train_mask = df["season"] < 2024
test_mask = df["season"] == 2024

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Przygotowanie danych dla stage 2
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. Wspólne modele bazowe
base_estimators = [
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    )),
    ("gbc", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
]

# === 5. Trzy stackingi

# LogisticRegression (ze scalerem)
stack_logreg = StackingClassifier(
    estimators=base_estimators,
    final_estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            multi_class="multinomial",
            solver="lbfgs",
            random_state=42
        )
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1
)

# MLPClassifier
stack_mlp = StackingClassifier(
    estimators=base_estimators,
    final_estimator=make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1
)

# XGBClassifier jako meta-model
stack_xgb = StackingClassifier(
    estimators=base_estimators,
    final_estimator=XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1
)

# === 6. VotingClassifier z 3 stackingów
voting_on_stackings = VotingClassifier(
    estimators=[
        ("stack_logreg", stack_logreg),
        ("stack_mlp", stack_mlp),
        ("stack_xgb", stack_xgb)
    ],
    voting="soft"
)

# === 7. Trening + predykcja
voting_on_stackings.fit(X_train_stage2, y_train_stage2)
probas_stage2 = voting_on_stackings.predict_proba(X_test_stage2)

# === 8. Tworzenie piątek
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 9. Zapis JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)



<<<<<<< local


/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:43:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:43:29] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[23:43:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[23:43:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[23:43:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_c

/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:15:04] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:15:05] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[13:15:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[13:15:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[13:15:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

/home/mateusz/VScode/PROJECT-NBA_Awards_Prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_c

>>>>>>> remote


In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV

# === 1. Wczytanie danych ===
df = pd.read_csv("nba_dataset_2010_2023.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

df["target_binary"] = (df["target"] > 0).astype(int)
train_mask = df["season"] < 2023
test_mask = df["season"] == 2023

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]

players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 2. Stage 1: klasyfikacja binarna
model_bin = XGBClassifier(
    objective="binary:logistic",
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
stage1_preds = model_bin.predict(X_test)

# === 3. Stage 2 dane
class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
inverse_class_map = {v: k for k, v in class_map.items()}
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

players_stage2 = players_test[stage1_mask].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

# === 4. Składniki stackingu
model_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

model_gbc = GradientBoostingClassifier(random_state=42)

logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    )
)

stacking_model = StackingClassifier(
    estimators=[
        ("xgb", model_xgb),
        ("gbc", model_gbc)
    ],
    final_estimator=logreg_pipeline,
    stack_method="predict_proba",
    passthrough=False,
    cv=3,
    n_jobs=-1
)

# === 5. RandomizedSearchCV na stackingu
param_distributions = {
    "final_estimator__logisticregression__C": [0.001, 0.01, 0.1, 1, 5, 10],
    "gbc__n_estimators": [100, 200, 300, 500],
    "gbc__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gbc__max_depth": [3, 4, 5, 6],
    "gbc__subsample": [0.7, 0.8, 1.0],
    "xgb__n_estimators": [200, 300, 500],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 1, 5]
}

search = RandomizedSearchCV(
    estimator=stacking_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train_stage2, y_train_stage2)
best_stack = search.best_estimator_

# === 6. Niezależny model: XGBClassifier
model_xgb_solo = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    use_label_encoder=False,
    eval_metric="mlogloss",
    n_estimators=300,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
model_xgb_solo.fit(X_train_stage2, y_train_stage2)

# === 7. VotingClassifier (stacking + XGB)
voting_combo = VotingClassifier(
    estimators=[
        ("stacking", best_stack),
        ("xgb_solo", model_xgb_solo)
    ],
    voting="soft",
    weights=[2, 1]  # większe zaufanie do stackingu
)

voting_combo.fit(X_train_stage2, y_train_stage2)
probas_stage2 = voting_combo.predict_proba(X_test_stage2)

# === 8. Tworzenie piątek
df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 9. Zapis JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

